# Random Slopes for Subject-Specific Trends

**Topics:** Random Slopes, Varying Effects, Growth Curves

## Overview

Extend random intercepts to include subject-specific slopes, allowing each individual to have their own trajectory.

## What You'll Learn

- Random slopes model specification
- Interpret slope variance
- Visualize individual trajectories
- Compare random intercepts vs random slopes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models.gamm import fit_gamm

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Data with Varying Slopes

In [ ]:
n_subjects = 25
n_timepoints = 6
n = n_subjects * n_timepoints

# Subject-level random effects
random_intercepts = np.random.randn(n_subjects) * 2
random_slopes = np.random.randn(n_subjects) * 0.5  # Individual time effects

data = []
for subj in range(n_subjects):
    for time in range(n_timepoints):
        # Each subject has own intercept and slope
        y = (
            10  # population intercept
            + 1.5 * time  # population slope
            + random_intercepts[subj]  # subject intercept deviation
            + random_slopes[subj] * time  # subject slope deviation
            + np.random.randn() * 1  # residual
        )
        
        data.append({
            'subject': subj,
            'time': time,
            'y': y,
            'true_intercept': 10 + random_intercepts[subj],
            'true_slope': 1.5 + random_slopes[subj]
        })

df = pd.DataFrame(data)

print(f"Generated {n} observations from {n_subjects} subjects")
print(f"\nRandom intercept SD: {random_intercepts.std():.2f}")
print(f"Random slope SD: {random_slopes.std():.2f}")

## Visualize Individual Trajectories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Individual trajectories
for subj in df['subject'].unique():
    subj_data = df[df['subject'] == subj]
    axes[0].plot(subj_data['time'], subj_data['y'], 
                alpha=0.4, linewidth=1.5, color='steelblue')

# Population mean
mean_trajectory = df.groupby('time')['y'].mean()
axes[0].plot(mean_trajectory.index, mean_trajectory.values, 
            color='red', linewidth=3, marker='o', markersize=8, label='Population mean')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Outcome')
axes[0].set_title('Individual Growth Curves (Random Slopes)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: True slopes distribution
true_slopes = df.groupby('subject')['true_slope'].first()
axes[1].hist(true_slopes, bins=15, edgecolor='k', alpha=0.7, color='steelblue')
axes[1].axvline(1.5, color='red', linestyle='--', linewidth=2, label='Population slope')
axes[1].set_xlabel('Subject-Specific Slope')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Individual Slopes')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nObservation: Different subjects have different growth rates!")

## Model 1: Random Intercepts Only

In [ ]:
# Model 1: Random Intercepts Only
result_ri = fit_gamm(
    formula='y ~ time + (1 | subject)',
    data=df,
    family='gaussian',
    covariance='identity'
)

print("Model 1: Random Intercepts Only")
print(f"\nFixed Effects:")
print(f"  Intercept: {result_ri.beta_parametric[0]:.2f}")
print(f"  Time: {result_ri.beta_parametric[1]:.2f}")
print(f"\nRandom Effects:")
# variance_components is now a list - [0] gives first term's covariance matrix
intercept_var = result_ri.variance_components[0][0, 0]
print(f"  Intercept SD: {np.sqrt(intercept_var):.2f}")
print(f"  Residual SD: {np.sqrt(result_ri.residual_variance):.2f}")

## Model 2: Random Intercepts + Random Slopes

In [ ]:
# Model 2: Random Intercepts + Random Slopes
result_ris = fit_gamm(
    formula='y ~ time + (1 + time | subject)',
    data=df,
    family='gaussian',
    covariance='unstructured'
)

print("\nModel 2: Random Intercepts + Random Slopes")
print(f"\nFixed Effects:")
print(f"  Intercept: {result_ris.beta_parametric[0]:.2f}")
print(f"  Time: {result_ris.beta_parametric[1]:.2f}")

# variance_components is now a list - [0] gives first term's covariance matrix
print(f"\nRandom Effects Covariance Matrix:")
print(result_ris.variance_components[0])
print(f"\nRandom Effects:")
intercept_var = result_ris.variance_components[0][0, 0]
slope_var = result_ris.variance_components[0][1, 1]
covar = result_ris.variance_components[0][0, 1]
print(f"  Intercept SD: {np.sqrt(intercept_var):.2f}")
print(f"  Slope SD: {np.sqrt(slope_var):.2f}")
print(f"  Correlation: {covar / (np.sqrt(intercept_var) * np.sqrt(slope_var)):.3f}")
print(f"  Residual SD: {np.sqrt(result_ris.residual_variance):.2f}")

# Extract random effects
# random_effects is a dict: {'subject': {subject_id: [intercept, slope], ...}}
subject_effects = result_ris.random_effects['subject']
print(f"\nRandom effects: {len(subject_effects)} subjects with 2 effects each (intercept, slope)")

## Compare Models

In [ ]:
# Predictions - use fitted values from the models
pred_ri = result_ri.fitted_values
pred_ris = result_ris.fitted_values

# Get y from dataframe (since it was overwritten in the data generation loop)
y_array = df['y'].values

# RMSE
from aurora.validation.metrics import root_mean_squared_error
rmse_ri = root_mean_squared_error(y_array, pred_ri)
rmse_ris = root_mean_squared_error(y_array, pred_ris)

print("Model Comparison:")
print(f"\n{'Model':<30} {'RMSE':<10} {'Residual SD':<15}")
print("="*55)
print(f"{'Random Intercepts':<30} {rmse_ri:<10.3f} {np.sqrt(result_ri.residual_variance):<15.3f}")
print(f"{'Random Intercepts + Slopes':<30} {rmse_ris:<10.3f} {np.sqrt(result_ris.residual_variance):<15.3f}")
print("\nRandom slopes model better captures individual variation")

## Visualize Fitted Trajectories

In [ ]:
# Add predictions to dataframe
df['pred_ri'] = pred_ri
df['pred_ris'] = pred_ris

# Sample 5 subjects for visualization
sample_subjects = df['subject'].unique()[:5]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for subj in sample_subjects:
    subj_data = df[df['subject'] == subj]
    
    # Left: Random intercepts only
    axes[0].scatter(subj_data['time'], subj_data['y'], alpha=0.6, s=50)
    axes[0].plot(subj_data['time'], subj_data['pred_ri'], linewidth=2)
    
    # Right: Random slopes
    axes[1].scatter(subj_data['time'], subj_data['y'], alpha=0.6, s=50)
    axes[1].plot(subj_data['time'], subj_data['pred_ris'], linewidth=2)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Outcome')
axes[0].set_title('Random Intercepts Only\n(Parallel trajectories)')
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Time')
axes[1].set_ylabel('Outcome')
axes[1].set_title('Random Intercepts + Slopes\n(Varying trajectories)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRandom slopes allow non-parallel trajectories")
print("Better fit when growth rates truly vary across subjects")

## Recover True Parameters

In [ ]:
# Extract estimated random effects from the dictionary
# random_effects['subject'] is a dict mapping subject_id -> [intercept, slope]
subject_effects = result_ris.random_effects['subject']

# Convert to arrays for comparison
estimated_intercepts = np.array([subject_effects[i][0] for i in range(n_subjects)])
estimated_slopes = np.array([subject_effects[i][1] for i in range(n_subjects)])

# Compare estimated vs true random effects
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Intercepts
axes[0].scatter(random_intercepts, estimated_intercepts, alpha=0.6, s=50, edgecolor='k')
lim = max(abs(random_intercepts).max(), abs(estimated_intercepts).max()) * 1.1
axes[0].plot([-lim, lim], [-lim, lim], 'r--', lw=2, label='Perfect recovery')
axes[0].set_xlabel('True Random Intercept')
axes[0].set_ylabel('Estimated Random Intercept')
axes[0].set_title('Random Intercept Recovery')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].axis('equal')

# Slopes
axes[1].scatter(random_slopes, estimated_slopes, alpha=0.6, s=50, edgecolor='k')
lim = max(abs(random_slopes).max(), abs(estimated_slopes).max()) * 1.1
axes[1].plot([-lim, lim], [-lim, lim], 'r--', lw=2, label='Perfect recovery')
axes[1].set_xlabel('True Random Slope')
axes[1].set_ylabel('Estimated Random Slope')
axes[1].set_title('Random Slope Recovery')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

# Correlation
corr_int = np.corrcoef(random_intercepts, estimated_intercepts)[0, 1]
corr_slope = np.corrcoef(random_slopes, estimated_slopes)[0, 1]

print(f"\nCorrelation (true vs estimated):")
print(f"  Intercepts: {corr_int:.3f}")
print(f"  Slopes: {corr_slope:.3f}")
print(f"\nModel successfully recovers individual-level parameters!")

## When to Use Random Slopes

Use random slopes when:
- Subjects have different trajectories over time
- Growth/decline rates vary across individuals
- Parallel trajectories assumption violated
- Treatment effects vary by person

## Model Formula (lme4-style)

```R
# Random intercepts only
y ~ time + (1 | subject)

# Random intercepts + slopes
y ~ time + (1 + time | subject)
```

**Next:** `04_longitudinal/03_nested_random_effects.ipynb` for multi-level structure